<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_SUPPLY_CHAIN_ANT_COLONY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import math
import random

# Calculate distance between two locations
def calculate_distance(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


# Select the next shelf using ACO probability
def select_next(current, unvisited, pheromone, distance, alpha, beta):
    probabilities = []

    for city in unvisited:
        tau = pheromone[current][city] ** alpha
        eta = (1 / distance[current][city]) ** beta
        probabilities.append(tau * eta)

    total = sum(probabilities)
    probabilities = [p / total for p in probabilities]

    return random.choices(unvisited, weights=probabilities, k=1)[0]


# Calculate total route distance
def route_distance(route, distance):
    total = 0

    for i in range(len(route) - 1):
        total += distance[route[i]][route[i + 1]]

    return total


# Ant Colony Optimization
def ant_colony_optimization(coordinates, ants, iterations):
    n = len(coordinates)

    alpha = 1
    beta = 2
    evaporation = 0.5
    Q = 100

    # Create distance matrix
    distance = [[0] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j:
                distance[i][j] = calculate_distance(
                    coordinates[i], coordinates[j]
                )

    # Initialize pheromone matrix
    pheromone = [[1.0] * n for _ in range(n)]

    best_route = None
    best_distance = float('inf')

    for iteration in range(iterations):
        all_routes = []

        # Each ant constructs a route
        for ant in range(ants):
            current = 0
            route = [0]
            unvisited = list(range(1, n))

            while unvisited:
                next_city = select_next(
                    current, unvisited, pheromone,
                    distance, alpha, beta
                )

                route.append(next_city)
                unvisited.remove(next_city)
                current = next_city

            # Return to packing station
            route.append(0)

            total = route_distance(route, distance)
            all_routes.append((route, total))

            if total < best_distance:
                best_distance = total
                best_route = route[:]

        # Pheromone evaporation
        for i in range(n):
            for j in range(n):
                pheromone[i][j] *= (1 - evaporation)

        # Pheromone deposition
        for route, total in all_routes:
            deposit = Q / total

            for i in range(len(route) - 1):
                a = route[i]
                b = route[i + 1]

                pheromone[a][b] += deposit
                pheromone[b][a] += deposit

    return best_route, best_distance


# Main program
random.seed(42)

n = int(input("Enter number of shelves: "))

coordinates = []

print("Enter coordinates (x y) for each location:")
print("Location 0 is the packing station.")

for i in range(n):
    x, y = map(float, input().split())
    coordinates.append((x, y))

ants = int(input("Enter number of ants: "))
iterations = int(input("Enter number of iterations: "))

best_route, best_distance = ant_colony_optimization(
    coordinates, ants, iterations
)

print("\nBest Picking Route:")

for location in best_route:
    if location == 0:
        print("Packing Station", end="")
    else:
        print("Shelf " + str(location), end="")

    if location != best_route[-1]:
        print(" -> ", end="")

print("\nMinimum Picking Distance:", round(best_distance, 2))


Enter number of shelves: 5
Enter coordinates (x y) for each location:
Location 0 is the packing station.
0 0
2 3
5 4
6 1
1 5
Enter number of ants: 10
Enter number of iterations: 100

Best Picking Route:
Packing StationShelf 3 -> Shelf 2 -> Shelf 4 -> Shelf 1 -> Packing Station
Minimum Picking Distance: 19.21
